# OverlapRL — Async GRPO Training Pipeline

**Staleness-aware scheduling + compute overlap for single-GPU RL post-training.**

> Set runtime to **A100 GPU**: Runtime → Change runtime type → A100 GPU

## 1. Setup

In [ ]:
import os, sys
if not os.path.exists('OverlapRL'):
    !git clone https://github.com/shiloz-stack/OverlapRL.git
else:
    !cd OverlapRL && git pull
os.chdir('OverlapRL')
sys.path.insert(0, '.')

In [ ]:
!pip install -q transformers torch pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU! Change runtime to A100.')

## 2. Run Tests

In [ ]:
!python -m pytest tests/ -v --tb=short

## 3. Benchmark: Sync vs Async vs Staleness-Aware vs Full

Each mode trains for 10 steps on Qwen2.5-0.5B with GSM8K math problems.

In [ ]:
# Run sync mode
!python scripts/benchmark.py --model Qwen/Qwen2.5-0.5B --steps 10 --mode sync --output sync_results.json

In [ ]:
# Run async mode
!python scripts/benchmark.py --model Qwen/Qwen2.5-0.5B --steps 10 --mode async --output async_results.json

In [ ]:
# Run async_stale mode
!python scripts/benchmark.py --model Qwen/Qwen2.5-0.5B --steps 10 --mode async_stale --output async_stale_results.json

In [ ]:
# Run full mode
!python scripts/benchmark.py --model Qwen/Qwen2.5-0.5B --steps 10 --mode full --output full_results.json

## 4. Visualize Results

In [ ]:
import json
import matplotlib.pyplot as plt

results = {}
for mode in ['sync', 'async', 'async_stale', 'full']:
    fname = f'{mode}_results.json'
    if os.path.exists(fname):
        with open(fname) as f:
            results[mode] = json.load(f)

modes = list(results.keys())
step_times = [results[m]['mean_step_time'] for m in modes]
wall_times = [results[m]['wall_time_s'] for m in modes]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(modes, step_times, color=['#4C72B0', '#55A868', '#C44E52', '#8172B2'])
axes[0].set_title('Mean Step Time (s)')
axes[0].tick_params(axis='x', rotation=15)
axes[1].bar(modes, wall_times, color=['#4C72B0', '#55A868', '#C44E52', '#8172B2'])
axes[1].set_title('Total Wall Time (s)')
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig('benchmark_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
first_rewards = [results[m].get('first_10_reward', 0) for m in modes]
last_rewards = [results[m].get('last_10_reward', 0) for m in modes]

x = range(len(modes))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - width/2 for i in x], first_rewards, width, label='First 10 steps', color='#4C72B0')
ax.bar([i + width/2 for i in x], last_rewards, width, label='Last 10 steps', color='#55A868')
ax.set_xticks(list(x))
ax.set_xticklabels(modes, rotation=15)
ax.set_ylabel('Mean Reward')
ax.set_title('Reward Improvement')
ax.legend()
plt.tight_layout()
plt.savefig('reward_improvement.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Takeaways

- **Async overlap** provides throughput improvement when generation is the bottleneck
- **Staleness scheduling** prevents training degradation from stale samples
- **Reference precompute** leverages generation's idle compute

Repo: https://github.com/shiloz-stack/OverlapRL